## Phase 1 : Importation et Exploration Initiale des Données

Dans cette première étape, nous mettons en place notre environnement de travail et chargeons les tables clés de la base de données Olist. L'objectif est de comprendre la structure brute des données avant d'appliquer le moindre filtre.

### 1. Importation des outils d'analyse


In [5]:

import pandas as pd

### 2. Chargement des fichiers sources (Data Ingestion)

In [6]:
df_orders = pd.read_csv("data/olist_orders_dataset.csv")
df_customers = pd.read_csv("data/olist_customers_dataset.csv")
df_reviews = pd.read_csv("data/olist_order_reviews_dataset.csv")

### 3. Vérification de la volumétrie

In [7]:
print(f"Commandes : {df_orders.shape}")
print(f"Clients : {df_customers.shape}")
print(f"Avis : {df_reviews.shape}")

Commandes : (99441, 8)
Clients : (99441, 5)
Avis : (99224, 7)


### 4. Inspection de la structure et des types de données (Data Profiling)

In [8]:
display(df_orders.head(3))
print("\nInformations sur les colonnes des commandes :")
df_orders.info()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00



Informations sur les colonnes des commandes :
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 21.9 MB


## Phase 2 : Data Cleaning et Typage des Données

Une analyse logistique fiable repose sur des calculs de durées précis. Les dates importées étant initialement formatées comme du texte brut, cette phase vise à corriger les types de données et à évaluer la complétude de notre jeu de données.

### 1. Conversion Temporelle (`pd.to_datetime`)
Nous identifions les 5 jalons clés du parcours d'une commande (de l'achat à la livraison estimée et réelle). Nous appliquons la méthode `pd.to_datetime()`[cite: 1] de Pandas pour transformer ces chaînes de caractères en objets temporels, permettant ainsi les calculs d'intervalles (deltas).

### 2. Audit des Valeurs Manquantes (`isnull`)
L'agrégation `.isnull().sum()` nous donne une cartographie immédiate de la qualité des données.
> 💡 **Perspectives Métier :** La présence de valeurs nulles dans des colonnes comme `order_delivered_customer_date` n'est pas une "erreur" informatique à supprimer aveuglément. Cela reflète une réalité opérationnelle : ces commandes sont probablement soit en cours d'acheminement, soit annulées. Une suppression brute fausserait notre taux de réussite logistique.

In [9]:
# 1. Liste des colonnes contenant des dates temporelles
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

# 2. Conversion optimisée de ces colonnes en datetime
for col in date_columns:
    df_orders[col] = pd.to_datetime(df_orders[col])

# 3. Vérification des nouveaux types
print("Types de données après conversion :")
print(df_orders[date_columns].dtypes)

# 4. Identification des données manquantes (Missing Values)
print("\nQuantité de valeurs manquantes par colonne :")
display(df_orders.isnull().sum())

Types de données après conversion :
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

Quantité de valeurs manquantes par colonne :


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

## Phase 3 : Modélisation et Ingénierie des Caractéristiques (Feature Engineering)

Pour répondre à notre problématique d'optimisation de la chaîne d'approvisionnement, nous devons structurer une table analytique (Master Table) prête pour la visualisation.

### 1. Filtrage Opérationnel
Nous isolons les commandes ayant atteint leur destination finale (`order_status == 'delivered'`). Les annulations et retours nécessiteraient une analyse distincte du taux d'attrition logistique. L'utilisation de `.copy()` assure que nous travaillons sur un sous-ensemble indépendant en mémoire.

### 2. Création de KPIs (Feature Engineering)
La valeur brute d'un horodatage est peu exploitable pour un décideur. Nous calculons un indicateur tangible : **le délai de livraison en jours** (`delivery_time_days`), en soustrayant la date d'achat de la date de livraison via l'accesseur `.dt.days`. 

### 3. Modélisation Relationnelle (Data Merging)
Nous consolidons notre vue à 360 degrés de la commande en utilisant `pd.merge()`[cite: 1] :
* **Inner Join avec `df_customers` :** Pour rapatrier la géographie du client (ville, état) via la clé `customer_id`[cite: 1].
* **Left Join avec `df_reviews` :** Pour associer le score de satisfaction (1 à 5) via la clé `order_id`[cite: 1]. L'utilisation d'une jointure à gauche garantit la conservation des commandes livrées même si le client n'a pas laissé d'évaluation.

In [11]:
# 1. Filtre métier : Conservation uniquement des commandes livrées
df_delivered = df_orders[df_orders['order_status'] == 'delivered'].copy()

# 2. Feature Engineering : Calcul du délai de livraison réel (en jours)
df_delivered['delivery_time_days'] = (
    df_delivered['order_delivered_customer_date'] - 
    df_delivered['order_purchase_timestamp']
).dt.days

# 3. Modélisation : Jointures des tables (Orders + Customers + Reviews)
# Jointure interne (inner) avec les clients
df_master = pd.merge(
    df_delivered, 
    df_customers, 
    on='customer_id', 
    how='inner'
)

# Jointure à gauche (left) avec les avis (car certaines commandes n'ont pas d'avis)
df_master = pd.merge(
    df_master, 
    df_reviews, 
    on='order_id', 
    how='left'
)

# 4. Vérification du jeu de données final
print(f"Dimensions de la table analytique finale : {df_master.shape}")
display(df_master[['order_id', 'customer_city', 'delivery_time_days', 'review_score']].head())

Dimensions de la table analytique finale : (97007, 19)


,order_id,customer_city,delivery_time_days,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,sao paulo,8.0,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,barreiras,13.0,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,vianopolis,9.0,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,sao goncalo do amarante,13.0,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,santo andre,2.0,5.0


## Phase 4 : Exportation des Données (Data Export)

Une fois le jeu de données nettoyé, enrichi de nouveaux KPIs et consolidé via nos jointures, nous l'exportons vers un format plat (CSV). 

Cette étape marque la fin de notre pipeline de préparation (ETL) et crée notre **"Gold Table"**. C'est ce fichier unique, léger et optimisé, qui servira de source de données pour la création de notre tableau de bord exécutif, garantissant ainsi des performances de visualisation optimales.

In [12]:
# 5. Exportation de la table d'or (Gold Layer) pour la visualisation
df_master.to_csv('data/olist_master_dataset.csv', index=False)

print("Exportation réussie : le fichier 'olist_master_dataset.csv' est prêt dans le dossier 'data/'.")

Exportation réussie : le fichier 'olist_master_dataset.csv' est prêt dans le dossier 'data/'.
